# Studio 3D AI Endpoints — Headless GPU Server (Colab T4 / L4)

Ejecuta **`black-forest-labs/FLUX.2-klein-4B`** y **`microsoft/TRELLIS.2-4B`** sin interfaz gráfica pesada, cada uno en **su propio entorno aislado** con motores nativos C++/GGML CUDA (`stable-diffusion.cpp` y `trellis.cpp`), serializados por un **Gateway GPU-Lease** para no pasar nunca los **15 GB VRAM / 12.7 GB RAM** de Colab.

In [ ]:
# @title 1. Verificar GPU y clonar repo santiago96ml/studio-3d-ai-endpoints
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y -o Dpkg::Options::='--force-confold' cmake ninja-build curl git >/dev/null

import os
REPO_DIR = '/content/studio-3d-ai-endpoints'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/santiago96ml/studio-3d-ai-endpoints.git /content/studio-3d-ai-endpoints
else:
    !git -C /content/studio-3d-ai-endpoints pull --ff-only

%cd /content/studio-3d-ai-endpoints

In [ ]:
# @title 2. Instalar entornos aislados por modelo (FLUX.2-klein-4B + TRELLIS.2-4B + Gateway)
import os

MODE = 'all'  # @param ['all', 'flux2', 'trellis2']
FLUX2_QUANT = 'Q4_K_M'  # @param ['Q4_K_M', 'Q5_K_M', 'Q8_0']
TRELLIS2_QUANT = 'q4'  # @param ['q4', 'q8']
TRELLIS2_INCLUDE_TEXTURE = '0'  # @param ['0', '1']
STUDIO_API_TOKEN = ''  # @param {type:'string'}

os.environ['FLUX2_QUANT'] = FLUX2_QUANT
os.environ['TRELLIS2_QUANT'] = TRELLIS2_QUANT
os.environ['TRELLIS2_INCLUDE_TEXTURE'] = TRELLIS2_INCLUDE_TEXTURE
os.environ['STUDIO_API_TOKEN'] = STUDIO_API_TOKEN
os.environ['COLAB_MODE'] = MODE

if MODE in ('all', 'flux2'):
    !bash scripts/setup_flux2.sh
if MODE in ('all', 'trellis2'):
    !bash scripts/setup_trellis2.sh
!bash scripts/setup_gateway.sh
print('✅ Entornos listos para modo:', MODE)

In [ ]:
# @title 3. Levantar endpoints + túnel Cloudflare (copiá la URL https://*.trycloudflare.com)
import os
mode = os.environ.get('COLAB_MODE', 'all')
!python3 scripts/start_endpoints.py --mode {mode}

In [ ]:
# @title 4. (Opcional) Prueba local rápida contra los endpoints corriendo en Colab
# Para ejecutar esto, primero corré la Celda 3 en segundo plano o probalo desde otra sesión/PC contra tu URL Cloudflare.
!curl -s http://127.0.0.1:8000/health | python3 -m json.tool